<a href="https://colab.research.google.com/github/nikitask14/pytorch-engineering-to-federated-learning/blob/main/Sitting19_Model_State.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import torch
import torch.nn as nn

In [34]:
class MyModel(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1 = nn.Linear(6,4)
    self.layer2 = nn.Linear(4,3)

  def forward(self, x):
    x = torch.relu(self.layer1(x))
    x = self.layer2(x)

    return x
model = MyModel()

Two models can have exactly the same architecture:

nn.Linear(6,4)


nn.Linear(4,3)

but have different learned states because their weights and biases contain different numbers.

###**Inspect the model state**

In [35]:
# model_state is just a variable name referring to the model’s state dictionary.
model_state = model.state_dict()

print(model_state["layer1.weight"])
print(model.state_dict()["layer1.weight"])

tensor([[ 0.0888,  0.3749,  0.2669,  0.0196,  0.0975,  0.1111],
        [ 0.0037, -0.1087, -0.1766, -0.3222,  0.2994, -0.0011],
        [ 0.3370,  0.0594,  0.3973,  0.0990,  0.0777, -0.2274],
        [-0.1004, -0.1917, -0.3025, -0.1435, -0.2082, -0.2236]])
tensor([[ 0.0888,  0.3749,  0.2669,  0.0196,  0.0975,  0.1111],
        [ 0.0037, -0.1087, -0.1766, -0.3222,  0.2994, -0.0011],
        [ 0.3370,  0.0594,  0.3973,  0.0990,  0.0777, -0.2274],
        [-0.1004, -0.1917, -0.3025, -0.1435, -0.2082, -0.2236]])


model.state_dict() gives us the dictionary-like state of the model.

Conceptually:

{

    "layer1.weight": tensor(...),

    "layer1.bias": tensor(...),

    "layer2.weight": tensor(...),

    "layer2.bias": tensor(...)
}


---------------


state["layer1.weight"]

means:

Look inside the dictionary-like object state and retrieve the value stored under the key "layer1.weight".

In [24]:
for name, tensor in model.state_dict().items():
  print(name, tensor.shape)

layer1.weight torch.Size([4, 6])
layer1.bias torch.Size([4])
layer2.weight torch.Size([3, 4])
layer2.bias torch.Size([3])



Conceptually, PyTorch packaged the current state of the model as:

"layer1.weight" → tensor of shape (4, 6)

"layer1.bias"   → tensor of shape (4,)

"layer2.weight" → tensor of shape (3, 4)

"layer2.bias"   → tensor of shape (3,)

The strings on the left are the dictionary keys. The tensors on the right are the values.

So state_dict() gave us access to the actual parameter tensors that describe the current state of this model.

In [36]:
print(model_state.keys())

odict_keys(['layer1.weight', 'layer1.bias', 'layer2.weight', 'layer2.bias'])


###**Inspect Tensors**

In [40]:
print(model_state["layer1.weight"])
print(model_state["layer1.weight"].shape)
print(model_state["layer1.bias"])
print(model_state["layer1.bias"].shape)

tensor([[ 0.0888,  0.3749,  0.2669,  0.0196,  0.0975,  0.1111],
        [ 0.0037, -0.1087, -0.1766, -0.3222,  0.2994, -0.0011],
        [ 0.3370,  0.0594,  0.3973,  0.0990,  0.0777, -0.2274],
        [-0.1004, -0.1917, -0.3025, -0.1435, -0.2082, -0.2236]])
torch.Size([4, 6])
tensor([ 0.2997,  0.2783, -0.2103, -0.0653])
torch.Size([4])


###**Two independent models**

In [41]:
# Can we have two separate model objects with the same architecture
#but their own independent parameter values?

model_A = MyModel()
model_B = MyModel()

In [45]:
model_A_state = model_A.state_dict()
model_B_state = model_B.state_dict()
print(model_A_state["layer1.weight"])
print(model_B_state["layer1.weight"])


tensor([[ 0.0465, -0.1125,  0.3493,  0.2142,  0.3680,  0.3268],
        [-0.3993, -0.1352,  0.0026, -0.0313, -0.0222,  0.2469],
        [ 0.2440, -0.3551, -0.0820, -0.1544, -0.1216, -0.0229],
        [ 0.0150, -0.0991, -0.3495,  0.3738, -0.3574,  0.4079]])
tensor([[-0.2604,  0.3665,  0.0177, -0.2311, -0.3025,  0.2996],
        [ 0.2298, -0.2487,  0.1682,  0.2188,  0.0587,  0.3149],
        [ 0.2750,  0.1686, -0.0037,  0.1443, -0.3717, -0.2622],
        [ 0.1513,  0.4027, -0.2295, -0.2871, -0.3586, -0.1287]])


Both layer1.weight tensors have shape (4, 6), so the architecture is the same, but the numerical values are clearly different.

###**Compare their states**

In [48]:
torch.equal(model_A_state["layer1.weight"], model_B_state["layer1.weight"])

False

**torch.equal()** checks both shape and numerical values.

So:

torch.equal(tensor_A, tensor_B)

It returns True only **when the tensors have the same shape, and
every corresponding element has the same numerical value**.

Now extending the same idea to all parameters. Instead of manually repeating that for:

layer1.weight

layer1.bias

layer2.weight

layer2.bias

We use a loop.

In [50]:
for name in model_A_state:
  print(name, torch.equal(model_A_state[name], model_B_state[name]))

layer1.weight False
layer1.bias False
layer2.weight False
layer2.bias False


###**load_state_dict()**

In [52]:
model_B.load_state_dict(model_A_state)

<All keys matched successfully>

In [53]:
for name in model_B_state:
  print(torch.equal(model_B_state[name], model_A_state[name]))

True
True
True
True


In [56]:
print(model_A_state["layer1.weight"])
print("-----------------------")
print(model_B_state["layer1.weight"])

tensor([[ 0.0465, -0.1125,  0.3493,  0.2142,  0.3680,  0.3268],
        [-0.3993, -0.1352,  0.0026, -0.0313, -0.0222,  0.2469],
        [ 0.2440, -0.3551, -0.0820, -0.1544, -0.1216, -0.0229],
        [ 0.0150, -0.0991, -0.3495,  0.3738, -0.3574,  0.4079]])
-----------------------
tensor([[ 0.0465, -0.1125,  0.3493,  0.2142,  0.3680,  0.3268],
        [-0.3993, -0.1352,  0.0026, -0.0313, -0.0222,  0.2469],
        [ 0.2440, -0.3551, -0.0820, -0.1544, -0.1216, -0.0229],
        [ 0.0150, -0.0991, -0.3495,  0.3738, -0.3574,  0.4079]])
